# BS5 NEW experiment: curriculum 0 to total60k
C-R/C-N, seeds551/552. 240k NEW training steps. Source dataset only; no parent/resume24k required. Not a historical continuation. See BS5_CURRICULUM_FRESH_60K_KAGGLE.md.


In [ ]:
import os, sys, json, pathlib, subprocess, hashlib
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
INPUT = pathlib.Path('/kaggle/input')
WORK = pathlib.Path('/kaggle/working')
sources = sorted({p.parents[2] for p in INPUT.rglob('run_bs5_curriculum_fresh_60k.py') if p.parent.name == 'training'})
assert len(sources) == 1, f'Attach one NEW source dataset containing run_bs5_curriculum_fresh_60k.py: {sources}'
SRC = sources[0]
os.chdir(SRC)
OUT = WORK/'bs5_curriculum_fresh_60k'
RUNNER = SRC/'scripts/training/run_bs5_curriculum_fresh_60k.py'
def run_command(*args):
    subprocess.run([sys.executable,'-B',str(RUNNER),*map(str,args)],cwd=SRC,check=True)
print('cd:', SRC, 'output:', OUT)
print('FRESH 0->60k; no parent dataset required; C-R/C-N x seeds551/552 only.')


In [ ]:
# Fresh initialization uses the installed runtime; freeze it for subsequent resume.
import torch, numpy as np
print(sys.version, torch.__version__, np.__version__, 'CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable GPU in Kaggle Notebook Settings.'
if not (OUT/'fresh_experiment.json').exists():
    assert not OUT.exists(), 'Partial prepare exists: preserve it and inspect before starting a new output.'
    run_command('prepare-fresh','--output',OUT,'--device','cuda')
else:
    print('Already prepared: preserve this fresh lineage; next cell resumes.')


In [ ]:
# Train NEW prefixes0->24k then continue their OWN full states to60k. No historical import.
# Re-run this cell to resume existing output; complete runs are skipped.
run_command('run','--output',OUT,'--resume')


In [ ]:
# Export the latest durable tail snapshot, then download archives and receipts.
import shutil
from IPython.display import FileLink, display
assert (OUT/'tail36k/protocol.json').is_file(), 'Prefixes not all complete yet. Keep prefix24k full output and resume cell3.'
run_command('export','--output',OUT)
TAIL = OUT/'tail36k'
receipts = list((TAIL/'exports').glob('receipt_*.json'))
assert receipts, 'No export receipt yet.'
receipt = max(receipts,key=lambda p:p.stat().st_mtime_ns)
info = json.loads(receipt.read_text())
for item in info['artifacts']:
    src = TAIL/item['path']
    digest = hashlib.sha256()
    with src.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b''):
            digest.update(chunk)
    assert digest.hexdigest() == item['sha256']
    target = WORK/src.name
    shutil.copyfile(src,target)
    display(FileLink(str(target)))
for src in (receipt, OUT/'fresh_budget_ledger.json'):
    target = WORK/src.name
    shutil.copyfile(src,target)
    display(FileLink(str(target)))
print('Download review_small, audit_light, repro_full, receipt and fresh_budget_ledger. Stop at60k.')
